# Model Architecture
ExciPick Heterogeneous GNN model components.

This notebook combines the individual components from the `model/` directory:
- `api_encoder`: API feature projector
- `strength_encoder`: Encodes dose strength and unit
- `gnn_layers`: Heterogeneous GNN encoder (HetGNN)
- `excipient_scorer`: Excipient logit scorer
- `FULL_MODEL`: The complete assembled architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv

# Assuming CONFIG is loaded in the workspace from config.py
from config import CONFIG

## API Encoder (`api_encoder.py`)
Projects 20 molecular descriptors to GNN hidden dimension. The heavy representational lifting is done by the GNN layers.

In [ ]:
class APIProjector(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(CONFIG["api_in"], CONFIG["gnn_hidden"]),
            nn.LayerNorm(CONFIG["gnn_hidden"]),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

## Strength Encoder (`strength_encoder.py`)

In [ ]:
class StrengthEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.per_unit_emb = nn.Embedding(
            CONFIG["per_unit_vocab"],
            CONFIG["per_unit_emb"]
        )

        self.net = nn.Sequential(
            nn.Linear(1 + CONFIG["per_unit_emb"], CONFIG["strength_out"]),
            nn.ReLU()
        )

    def forward(self, dose, per_unit):
        emb = self.per_unit_emb(per_unit)
        x = torch.cat([dose.unsqueeze(1), emb], dim=1)
        return self.net(x)

## GNN Layers (`gnn_layers.py`)
Heterogeneous GNN encoder for ExciPick.
Uses PyG's HeteroConv with SAGEConv per edge type.

In [ ]:
class HeteroGNNEncoder(nn.Module):
    """
    Multi-layer heterogeneous GNN using SAGEConv per edge type.

    Args:
        metadata: tuple of (node_types, edge_types) from HeteroData.metadata()
        hidden_dim: hidden dimension for all layers
        num_layers: number of message passing rounds
        dropout: dropout rate
    """

    def __init__(self, metadata, hidden_dim, num_layers, dropout=0.2):
        super().__init__()

        self.num_layers = num_layers
        self.dropout = dropout

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            # One SAGEConv per edge type
            conv_dict = {}
            for edge_type in metadata[1]:
                conv_dict[edge_type] = SAGEConv((-1, -1), hidden_dim)

            self.convs.append(HeteroConv(conv_dict, aggr="sum"))

            # LayerNorm per node type
            norm_dict = nn.ModuleDict({
                node_type: nn.LayerNorm(hidden_dim)
                for node_type in metadata[0]
            })
            self.norms.append(norm_dict)

    def forward(self, x_dict, edge_index_dict):
        """
        Args:
            x_dict: {node_type: (num_nodes, hidden_dim)} node features
            edge_index_dict: {edge_type: (2, num_edges)} edge indices

        Returns:
            x_dict: enriched node embeddings, same structure as input
        """
        for conv, norm_dict in zip(self.convs, self.norms):
            # Message passing
            x_dict_new = conv(x_dict, edge_index_dict)

            # Residual + LayerNorm + ReLU + Dropout
            x_dict = {
                key: F.dropout(
                    F.relu(norm_dict[key](x_dict_new[key] + x_dict[key])),
                    p=self.dropout,
                    training=self.training,
                )
                for key in x_dict_new.keys()
            }

        return x_dict

## Excipient Scorer (`excipient_scorer.py`)
Scores each excipient against a formulation context. In the HetGNN version, excipient embeddings come from the GNN encoder (enriched via message passing), not from a local nn.Embedding.

In [ ]:
class Scorer(nn.Module):
    def __init__(self):
        super().__init__()

        input_dim = CONFIG["context_out"] + CONFIG["gnn_hidden"]

        self.net = nn.Sequential(
            nn.Linear(input_dim, CONFIG["scorer_hidden"]),
            nn.ReLU(),
            nn.Dropout(CONFIG["dropout_scorer"]),
            nn.Linear(CONFIG["scorer_hidden"], 1),
        )

    def forward(self, context, exc_embs):
        """
        Args:
            context:  (B, context_out) — fused formulation context
            exc_embs: (V, gnn_hidden)  — GNN-enriched excipient embeddings

        Returns:
            scores: (B, V) — one logit per excipient
        """
        B = context.shape[0]
        V = exc_embs.shape[0]

        # Tile context and excipient embeddings for pairwise scoring
        context_exp = context.unsqueeze(1).expand(-1, V, -1)   # (B, V, context_out)
        exc_exp = exc_embs.unsqueeze(0).expand(B, -1, -1)      # (B, V, gnn_hidden)

        x = torch.cat([context_exp, exc_exp], dim=2)            # (B, V, input_dim)
        scores = self.net(x).squeeze(-1)                        # (B, V)

        return scores

## Full Model (`FULL_MODEL.py`)
Architecture:
1. Project API features + excipient embeddings to GNN hidden dim
2. HetGNN message passing enriches all node embeddings
3. Per-sample: look up enriched API embedding + encode dose/route/form
4. Score enriched context against all enriched excipient embeddings

In [ ]:
class ExciPickHGNN(nn.Module):
    def __init__(self, graph_metadata, vocab_size):
        """
        Args:
            graph_metadata: tuple from HeteroData.metadata()
                            (node_types, edge_types)
            vocab_size: number of excipients in vocabulary
        """
        super().__init__()

        # --- Node feature projectors ---
        self.api_proj = APIProjector()
        self.exc_emb = nn.Embedding(vocab_size, CONFIG["gnn_hidden"])

        # --- GNN encoder ---
        self.gnn = HeteroGNNEncoder(
            metadata=graph_metadata,
            hidden_dim=CONFIG["gnn_hidden"],
            num_layers=CONFIG["gnn_layers"],
            dropout=CONFIG["gnn_dropout"],
        )

        # --- Dose encoder (unchanged from v1) ---
        self.strength_encoder = StrengthEncoder()

        # --- Route / Form embeddings ---
        self.route_emb = nn.Embedding(CONFIG["route_vocab"], CONFIG["route_emb"])
        self.form_emb = nn.Embedding(CONFIG["form_vocab"], CONFIG["form_emb"])

        # --- Context fusion ---
        fusion_in = (
            CONFIG["gnn_hidden"]
            + CONFIG["strength_out"]
            + CONFIG["route_emb"]
            + CONFIG["form_emb"]
        )

        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, CONFIG["context_out"]),
            nn.LayerNorm(CONFIG["context_out"]),
            nn.ReLU(),
            nn.Dropout(CONFIG["dropout_context"]),
        )

        # --- Excipient scorer ---
        self.scorer = Scorer()

    def forward(self, graph, api_idx, dose, per_unit, route, form):
        """
        Args:
            graph:    HeteroData with enriched node features
            api_idx:  (B,) long — indices of API nodes for this batch
            dose:     (B,) float — normalized dose
            per_unit: (B,) long — per-unit category ID
            route:    (B,) long — route category ID
            form:     (B,) long — dosage form category ID

        Returns:
            scores: (B, vocab_size) — raw logit scores per excipient
        """
        # 1. Project node features to GNN hidden dim
        x_dict = {
            "api": self.api_proj(graph["api"].x),
            "excipient": self.exc_emb.weight,
        }

        # 2. GNN message passing
        enriched = self.gnn(x_dict, graph.edge_index_dict)
        enriched_api = enriched["api"]           # (num_apis, gnn_hidden)
        enriched_exc = enriched["excipient"]     # (V, gnn_hidden)

        # 3. Look up this batch's API embeddings
        batch_api = enriched_api[api_idx]        # (B, gnn_hidden)

        # 4. Encode dose strength
        strength = self.strength_encoder(dose, per_unit)   # (B, strength_out)

        # 5. Encode route and form
        route_e = self.route_emb(route)          # (B, route_emb)
        form_e = self.form_emb(form)             # (B, form_emb)

        # 6. Fuse all context
        context = self.fusion(
            torch.cat([batch_api, strength, route_e, form_e], dim=1)
        )  # (B, context_out)

        # 7. Score against all enriched excipient embeddings
        scores = self.scorer(context, enriched_exc)   # (B, V)

        return scores